# Check Incorporated as BEN elligible businesses by querying the PROD database

In [ ]:
%pip install -q psycopg2-binary
%pip install pandas requests

import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

# this will load all the envars from a .env file located in the project root (api)
load_dotenv(find_dotenv())

%load_ext sql

In [ ]:
connect_to_db = 'postgresql://' + \
                os.getenv('ENTITY_DATABASE_USERNAME', '') + ":" + os.getenv('ENTITY_DATABASE_PASSWORD', '') +'@' + \
                os.getenv('ENTITY_DATABASE_HOST', '') + ':' + os.getenv('ENTITY_DATABASE_PORT', '5432') + '/' + \
                os.getenv('ENTITY_DATABASE_NAME', '');
engine = create_engine(connect_to_db)

# Test connection
with engine.connect() as conn:
    print(f"Connected successfully to {os.getenv('ENVIRONMENT', '')}")

<b>This section checks if any of the DRAFTS are completed and businesses are elligible to run the notebooks</b>

If business is elligible:
- check if AR is due (check directly in the UI by logging in as Staff).<br>
    - If AR is due, then Correction application cannot be filed. No further action required<br>
    - If no AR due, run Registrar's Notation first with `add_registrars_notation_ia.ipynb` notebook and then Corrections with `add_corrections_ia.ipynb`.<br>
- check if status has changed to HISTORICAL
    - no further action

In [ ]:
from businesses_with_drafts import businesses

elligible_businesses = []
non_elligible_businesses = []

# loop through list of businesses to create filing
with engine.connect() as conn:
    for identifier in businesses:
        draft_details_query = text("""
            SELECT b.state, 
                (SELECT COUNT(1) 
                    FROM filings f 
                    WHERE f.business_id = b.id 
                    AND f.status in ('DRAFT', 'PENDING')) <> 0 AS has_draft 
            FROM businesses b 
            WHERE b.identifier = :identifier
        """)
        draft_details_query_result = conn.execute(draft_details_query, {"identifier": identifier})
        draft_details = draft_details_query_result.mappings().fetchone()

        if draft_details:
            state = draft_details['state']
            has_draft = draft_details['has_draft']
        
        if not has_draft:
            elligible_businesses.append((identifier, state))
            continue

        if has_draft:
            non_elligible_businesses.append((identifier, state))
            continue
print('Eligible businesses to file Registrar Notation for:', elligible_businesses)
print('Non-eligible businesses to file Registrar Notation for:', non_elligible_businesses)

<b>This section checks if any of the HISTORICAL business became ACTIVE</b>



If business becomes ACTIVE:
- check if AR is due (check directly in the UI by logging in as Staff).<br>
    - If AR is due, then Correction application cannot be filed. No further action required<br>
    - If no AR due, run Registrar's Notation first with `add_registrars_notation_ia.ipynb` notebook and then Corrections with `add_corrections_ia.ipynb`.<br><br>

In [ ]:
from historical_businesses import businesses

elligible_businesses = []
non_elligible_businesses = []

# loop through list of businesses to create filing
with engine.connect() as conn:
    query = text("""
        SELECT b.identifier, b.state
        FROM businesses b
        WHERE b.identifier in :businesses
                    and b.state in ('ACTIVE')
    """)
    query_result = conn.execute(query, {"businesses": tuple(businesses)})
    print('Businesses in historical list which became active:', query_result.fetchall())